# Mechanistic (two-population) resistance — the resistance MODEL as a model-selection axis

**NOT FOR CLINICAL USE.** Population/trial-level forward simulation only; no individual prognosis, no clonal composition for a person, no therapy ranking. Executed in CI (nbmake).

Resistance is the central modeling challenge of oncology (the λ "Hydra" term). Onkos has modeled it *phenomenologically* (Claret: the drug effect decays as `e^{−λt}`). This notebook adds the *mechanistic* alternative — a drug-**sensitive** clone plus a pre-existing drug-**resistant** clone (Goldie-Coldman) — and shows that the *choice between the two resistance models* is a model-selection axis: they agree on the early trial window yet diverge in the tumor tail. See `docs/specs/research/mechanistic-resistance.md`.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import onkos
from scipy.integrate import solve_ivp
from onkos.export.reference import init_vector
from onkos.export.registry import get_kernel, kernel_values

ds = onkos.load()
ctx = {'tumor_type': 'NSCLC', 'line': 'first'}
rid = 'resistance.nsclc_first_line.two_population'
print(f'{len(ds)} records | onkos {onkos.__version__}')

## The clone decomposition

`dS/dt = (kg − kd·E)·S` (sensitive, killed), `dR/dt = kgr·R` (resistant, untouched), observed tumor `V = S + R`. The drug crushes the sensitive clone to a nadir; the resistant clone then outgrows. `R0` (the initial resistant burden) is biologically interpretable — unlike the phenomenological `λ`.

In [ ]:
t = np.linspace(0, 156, 313)
spec = get_kernel(ds[rid])
v = kernel_values(ds[rid])
v['V0'] = onkos.simulate(ds, rid, context=ctx, drug_effect=1.0, t=t).tumor_size[0] - v['R0']
v['E'] = 1.0
sol = solve_ivp(lambda tt, yy: spec.rhs(tt, yy, v), (t[0], t[-1]), init_vector(spec, v),
                t_eval=t, rtol=1e-8, atol=1e-10, method='LSODA')
s, r = sol.y

# Closed form: V(t) = V0 e^{(kg-kd E)t} + R0 e^{kgr t}
closed = v['V0']*np.exp((v['kg']-v['kd']*v['E'])*t) + v['R0']*np.exp(v['kgr']*t)
assert np.allclose(s + r, closed, rtol=1e-4)
print('closed form matches integration; late log-slope -> kgr:',
      round(np.polyfit(t[-40:], np.log((s+r)[-40:]), 1)[0], 4), '==', v['kgr'])

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(t, s, label='sensitive S (killed)')
ax.semilogy(t, r, label='resistant R (outgrows)')
ax.semilogy(t, s + r, 'k--', lw=2, label='observed V = S + R')
ax.set(xlabel='weeks', ylabel='tumor size (mm, SLD, log)', title='Two-population clone decomposition')
ax.legend(fontsize=8)
plt.show()

## The honest finding: agree early, diverge late

The mechanistic model is tuned to share the early kill with the phenomenological Claret model (matched `kd`). So they agree at **week 8** — and hence on the **week-8-driven OS** — yet diverge sharply in the tumor **tail**. The resistance-model risk is real but nearly invisible to a short-trial surrogate: exactly how a short-trial-fit resistance model transports silently.

In [ ]:
mech = onkos.simulate(ds, rid, context=ctx, drug_effect=1.0, t=t)
claret = onkos.simulate(ds, 'resistance.claret_2009.tgi', context=ctx, drug_effect=1.0, t=t)
print(f"week-8 change : mechanistic {mech.metrics['week8_relative_change']:+.2f}  "
      f"claret {claret.metrics['week8_relative_change']:+.2f}  -> nearly identical")
print(f"median OS     : mechanistic {mech.median_os:.0f}  claret {claret.median_os:.0f} wk -> close")
print(f"tumor at 3 yr : mechanistic {mech.tumor_size[-1]:.1f}  claret {claret.tumor_size[-1]:.1f} mm "
      f"-> {mech.tumor_size[-1]/claret.tumor_size[-1]:.1f}x split (the divergence the surrogate misses)")

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(t, claret.tumor_size, label='phenomenological (Claret λ)')
ax.semilogy(t, mech.tumor_size, label='mechanistic (resistant subclone)')
ax.axvline(8, ls=':', color='grey')
ax.set(xlabel='weeks', ylabel='tumor size (mm, SLD, log)', title='Same early kill, divergent tail')
ax.legend(fontsize=8)
plt.show()

## Guardrails: mechanistic does not mean measured

The interpretable resistant burden `R0` is still tier C and practically unidentifiable from a realistic trial (composes with the v0.22 identifiability analyzer); an out-of-context transport still floors to D; both compartments round-trip through SBML/NONMEM.

In [ ]:
# R0 is interpretable but practically unidentifiable from a realistic RECIST schedule.
idn = onkos.identifiability(ds, rid, context=ctx)
r0 = next(p for p in idn.params if p.symbol == 'R0')
print(f'R0 predicted RSE {r0.rse_percent:.0f}%  -> identifiable? {r0.identifiable}')
assert not r0.identifiable

# Out-of-context transport floors to D.
out = onkos.simulate(ds, rid, context={'tumor_type': 'breast', 'line': 'first'}, drug_effect=1.0)
print('transported tier:', out.tier)
assert out.tier == 'D'

# Both compartments export and round-trip.
from onkos.export.sbml import to_sbml
xml = to_sbml(ds[rid])
assert 'species id="sensitive"' in xml and 'species id="resistant"' in xml
print('SBML carries both clones; guardrails OK')